In [ ]:
%pip install google-genai

In [5]:
import os
from pathlib import Path

from google import genai
from google.genai import types

In [3]:
%pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [6]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY in your .env file.")

 

In [ ]:
client = genai.Client()  # uses GEMINI_API_KEY environment variable

# 1) Point to the PDFs (a folder, or a list of paths)
pdf_folder = Path("pdf_folder")               # folder next to your notebook in VS Code
pdf_paths = sorted(pdf_folder.glob("*.pdf"))
# or: pdf_paths = [Path("paper1.pdf"), Path("paper2.pdf")]

if not pdf_paths:
    raise FileNotFoundError(f"No PDFs found in {pdf_folder.resolve()}")

# 2) Upload them once
uploaded_files = [client.files.upload(file=p) for p in pdf_paths]
print("Uploaded:", [f.display_name or f.name for f in uploaded_files])

# 3) Create the chat with instructions to rely on the articles
chat = client.chats.create(
    model="gemini-3.8-flash",
    config=types.GenerateContentConfig(
        system_instruction=(
            "Answer using ONLY the provided PDF articles. "
            "Cite the article (file name) and page/section when possible. "
            "If the articles do not contain the answer, say so clearly."
        ),
        thinking_config=types.ThinkingConfig(
            thinking_budget=-1,
            include_thoughts=True,
        ),
    ),
)

# 4) First message: send the PDFs together with your prompt
prompt = """
You are advising a clinical guideline committee that is deciding whether inflammatory biomarkers should be incorporated into
 osteoporosis assessment. You have been provided several research papers examining the relationship between inflammation and bone health. 
 The evidence includes studies of the Systemic Immune-Inflammation Index, dietary inflammatory index, C-reactive protein, 
 neutrophil-to-lymphocyte ratio, and other inflammatory markers. Some studies report associations between inflammatory markers and reduced bone mineral density,
 while other analyses show weakened or non-significant relationships after adjustment for confounding variables. 
 Several analyses were conducted in populations with chronic kidney disease, postmenopausal women, and general adult populations. 
 A machine-learning study reported strong predictive performance when inflammatory markers were included in osteoporosis prediction models.

Your task is to answer the following decision question: Should inflammatory parameters, 
such as SII currently be used as clinical markers for osteoporosis?

Provide a recommendation, explain the supporting evidence, discuss contradictory findings, 
evaluate the strength and limitations of the evidence, address causality versus association, 
and explain whether inflammatory markers should be used as standalone markers, adjunctive markers, 
or research tools. Conclude with a clinical recommendation.
"""
response = chat.send_message([*uploaded_files, prompt])
# 5) Collect and print the answer and thoughts
answer_text = ""
thoughts_text = ""

for part in response.candidates[0].content.parts:
    if not part.text:
        continue
    if part.thought:
        thoughts_text += part.text + "\n"      # collected
        print("THOUGHTS:\n", part.text, "\n")  # printed
    else:
        answer_text += part.text + "\n"        # collected
        print("ANSWER:\n", part.text)          # printed

Uploaded: ['files/dxfg8s5e2h2k', 'files/xlrwip097975', 'files/0dw6tgu1hpvk', 'files/r0cd3i9cezrv', 'files/akai6pwhgl62', 'files/yvx9pmi7v828', 'files/doov2yqv5oqz', 'files/8ofk0n4vnwml', 'files/apa4h5har9u4', 'files/w1ehjax1n2xv', 'files/6eoedwv1r2z6', 'files/a0iff6cvg1as', 'files/me19b5q7424r', 'files/phgio65pmfcz', 'files/1m22jqpc0sig', 'files/qs963n9ir5pq', 'files/jp9lcoers5cf']
THOUGHTS:
 **Clinical Advisory Assessment: Inflammatory Parameters as Osteoporosis Markers**

Alright, let's break this down. The question is whether we can use inflammatory parameters, specifically looking at SII, as clinical markers for osteoporosis. My recommendation, based solely on this provided evidence, is a firm "no." We cannot currently recommend using these inflammatory markers as standalone diagnostic tools. Their role, at best, is as research tools and potential future adjunctive screening markers, pending further rigorous validation.

Here's the rationale:

*   **Evidence Supporting a Link**: Se

In [ ]:
# 6) Save to files
Path("prompt.txt").write_text(prompt, encoding="utf-8")
Path("answer.txt").write_text(answer_text, encoding="utf-8")
Path("thoughts.txt").write_text(thoughts_text, encoding="utf-8")

print("\nSaved: prompt.txt, answer.txt, thoughts.txt")